<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/1_evolution_strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%matplotlib inline

In [ ]:
from __future__ import division          # use // for integer division
from __future__ import absolute_import   # use from . import
from __future__ import print_function    # use print("...") instead of print "..."
from __future__ import unicode_literals  # all the strings are unicode

# 使用 Evolution Strategy 进行无约束连续优化

Evolution Strategy（ES，进化策略）是一类进化计算方法，也就是受生物进化机制启发的优化算法。目前，ES 中的 Covariance Matrix Adaptation Evolution Strategy（CMA-ES，协方差矩阵自适应进化策略）被广泛认为是性能最优秀的黑盒连续优化算法之一。

这里以一种简单的 Step-Size Adaptive ES（步长自适应进化策略）为例，理解 ES 的基本工作原理。

## 不更新步长的 ES

这里先去掉 ES 的绝大多数功能，只保留最基本的骨架，观察它会表现出怎样的行为。在后续 Notebook 中，我们再逐个考察现代 CMA-ES 中各个组件的作用。

下面实现的算法只是在不断重复以下随机搜索过程：

1. 初始化均值向量 $m$ 和步长 $\sigma$
2. 按照正态分布 $\mathcal{N}(m, \sigma^2 I)$，独立生成 $\lambda$ 个候选解 $x_i = m + \sigma \mathcal{N}(0, I)$（$i=1,\dots,\lambda$）
3. 评估每个候选解的目标函数值 $f(x_i)$；这些评估可以并行进行
4. 按目标函数值从小到大排序。记 $x_{i:\lambda}$ 为目标函数值第 $i$ 小的候选解
5. 按下式更新均值向量
$$
m \leftarrow m + \sum_{i=1}^{\lambda} w_i (x_{i:\lambda}-m)=\sum_{i=1}^{\lambda}w_i x_{i:\lambda}
$$
6. 返回步骤 2

其中，$w_1 \leq \dots \leq w_\lambda$ 是按候选解排名设置的权重。为便于理解，下面取 $\mu=\lfloor\lambda/4\rfloor$（选择的优秀候选解数量），并令 $w_1=\dots=w_\mu=1/\mu$，$w_{\mu+1}=\dots=w_\lambda=0$。现代 CMA-ES 通常使用随排名变化的非均匀权重，但它不会从根本上改变这里要观察的搜索行为，所以本例采用更简单的均匀权重。

因此，这个算法本质上只是不断从正态分布生成多个候选解，然后把其中排名前 $\mu$ 个解的平均值作为下一代正态分布的均值向量。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class ES(object):
    """不进行步长更新的 Evolution Strategy。"""

    def __init__(self, func, init_mean, init_sigma, nsample):
        """构造函数

        Parameters
        ----------
        func : callable
            目标函数（最小化）
        init_mean : ndarray (1D)
            初始均值向量
        init_sigma : float
            初始步长
        nsample : int
            样本数量
        """
        self.func = func
        self.mean = init_mean
        self.sigma = init_sigma
        self.N = self.mean.shape[0]                     # 搜索空间维数
        self.arx = np.zeros((nsample, self.N)) * np.nan # 候选解
        self.arf = np.zeros(nsample) * np.nan           # 候选解的目标函数值

        self.weights = np.zeros(nsample)
        self.weights[:nsample//4] = 1.0 / (nsample//4)  # 权重，总和为 1

    def sample(self):
        """生成候选解。"""
        self.arx = self.mean + self.sigma * np.random.normal(size=self.arx.shape)

    def evaluate(self):
        """评估候选解。"""
        for i in range(self.arf.shape[0]):
            self.arf[i] = self.func(self.arx[i])

    def update_mean(self):
        """更新均值向量。"""
        idx = np.argsort(self.arf)  # idx[i] 是目标函数值排名第 i 的候选解索引
        self.mean += np.dot(self.weights, (self.arx[idx] - self.mean))


In [ ]:
def sphere(x):
    """返回向量的范数，最优解为 (0,...,0)。"""
    return np.linalg.norm(x)

#### 使用 ES 优化 Sphere 函数

In [ ]:
es = ES(func=sphere,
        init_mean=np.ones(10),
        init_sigma=0.001,
        nsample=10)

maxiter = 5000
fbest = np.zeros(maxiter) * np.nan
fmean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
for i in range(fbest.shape[0]):
    es.sample()
    es.evaluate()
    es.update_mean()
    fbest[i] = es.arf.min()
    fmean[i] = sphere(es.mean)
    sigmaN[i] = es.sigma * es.N

#### 绘制结果

In [ ]:
plt.semilogy(fbest, '-r', label='f(best)')
plt.semilogy(fmean, '-b', label='f(mean)')
plt.semilogy(sigmaN, '--g', label='sigma*N')
plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend()

## 思考与实验

* 改变 `init_mean` 和 `init_sigma` 进行实验（必要时相应调整 `maxiter`）
* 将算法行为划分为三个阶段：前期搜索缓慢、中期搜索快速、后期搜索停滞。分别思考每个阶段中 `mean` 与 `sigma` 之间是什么关系
* 尝试实现步长更新，使中期这种快速搜索状态能够持续下去

In [ ]:
es = ES(func=sphere,
        init_mean=np.ones(10),
        init_sigma=0.001,
        nsample=10)

maxiter = 100
fbest = np.zeros(maxiter) * np.nan
fmean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
for i in range(fbest.shape[0]):
    es.sample()
    es.evaluate()
    es.update_mean()
    # 更新步长
    old_sigma = es.sigma

    es.sigma = sphere(es.mean) / es.N
    # 步长更新结束
    fbest[i] = es.arf.min()
    fmean[i] = sphere(es.mean)
    sigmaN[i] = es.sigma * es.N